## Python configuration

In [1]:
from neo4j import GraphDatabase
import pandas as pd
import numpy as np
import networkx as nx
import time

In [2]:
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "password"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

## Get the friendship graph from Neo4j

In [3]:
def load_friendship_graph(driver):
    query = """
    MATCH (s1:Student)-[:EST_AMI_DE]-(s2:Student)
    WHERE elementId(s1) < elementId(s2)
    RETURN s1.id AS source, s2.id AS target
    """

    G = nx.Graph()

    with driver.session() as session:
        result = session.run(query)

        for record in result:
            source = record["source"]
            target = record["target"]

            G.add_edge(source, target)

    return G

In [4]:
G = load_friendship_graph(driver)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1500
Edges: 7465


## Get community assignments

In [5]:
def get_communities(driver, property_name):

    query = f"""
    MATCH (s:Student)
    WHERE s.{property_name} IS NOT NULL
    RETURN s.id AS student_id,
           s.{property_name} AS community_id
    """

    communities = {}

    with driver.session() as session:
        result = session.run(query)

        for record in result:
            student_id = record["student_id"]
            community_id = record["community_id"]

            communities[student_id] = community_id

    return communities

In [6]:
louvain_communities = get_communities(
    driver,
    "louvainCommunity"
)

leiden_communities = get_communities(
    driver,
    "leidenCommunity"
)

lpa_communities = get_communities(
    driver,
    "labelPropagationCommunity"
)

## Calculate the number of communities

In [7]:
def number_of_communities(communities):

    return len(set(communities.values()))

In [8]:
print(
    "Louvain:",
    number_of_communities(louvain_communities)
)

print(
    "Leiden:",
    number_of_communities(leiden_communities)
)

print(
    "LPA:",
    number_of_communities(lpa_communities)
)

Louvain: 18
Leiden: 15
LPA: 1


## Calculate average community size

In [9]:
def community_size_statistics(communities):

    sizes = pd.Series(
        list(communities.values())
    ).value_counts()

    average_size = sizes.mean()

    standard_deviation = sizes.std(
        ddof=0
    )

    return average_size, standard_deviation

In [10]:
avg_louvain, std_louvain = community_size_statistics(
    louvain_communities
)

avg_leiden, std_leiden = community_size_statistics(
    leiden_communities
)

avg_lpa, std_lpa = community_size_statistics(
    lpa_communities
)

In [11]:
print(
    "Louvain:",
    avg_louvain,std_louvain
)

print(
    "Leiden:",
    avg_leiden,std_leiden
)

print(
    "LPA:",
    avg_lpa,std_lpa
)

Louvain: 83.33333333333333 28.146442443446066
Leiden: 100.0 50.428827734408685
LPA: 1500.0 0.0


## Calculate modularity Q

In [12]:
def calculate_modularity(G, communities):

    community_sets = {}

    for node, community_id in communities.items():

        if community_id not in community_sets:
            community_sets[community_id] = set()

        community_sets[community_id].add(node)

    community_list = list(
        community_sets.values()
    )

    return nx.community.modularity(
        G,
        community_list
    )

In [13]:
louvain_modularity = calculate_modularity(
    G,
    louvain_communities
)

leiden_modularity = calculate_modularity(
    G,
    leiden_communities
)

lpa_modularity = calculate_modularity(
    G,
    lpa_communities
)

In [14]:
print("Louvain Q:", louvain_modularity)
print("Leiden Q:", leiden_modularity)
print("LPA Q:", lpa_modularity)

Louvain Q: 0.31869690616940227
Leiden Q: 0.3239789165693531
LPA Q: 0.0


## Calculate conductance

In [15]:
def calculate_average_conductance(
    G,
    communities
):

    community_sets = {}

    for node, community_id in communities.items():

        if community_id not in community_sets:
            community_sets[community_id] = set()

        community_sets[community_id].add(node)

    conductances = []

    for community in community_sets.values():

        # A community containing the entire graph
        # has no boundary edges.
        if len(community) == G.number_of_nodes():
            continue

        conductance = nx.conductance(
            G,
            community
        )

        conductances.append(
            conductance
        )

    if len(conductances) == 0:
        return None

    return np.mean(conductances)

In [16]:
louvain_conductance = calculate_average_conductance(
    G,
    louvain_communities
)

leiden_conductance = calculate_average_conductance(
    G,
    leiden_communities
)

lpa_conductance = calculate_average_conductance(
    G,
    lpa_communities
)

In [17]:
print(
    "Louvain conductance:",
    louvain_conductance
)

print(
    "Leiden conductance:",
    leiden_conductance
)

print(
    "LPA conductance:",
    lpa_conductance
)

Louvain conductance: 0.623883004815982
Leiden conductance: 0.6057088148264409
LPA conductance: None


## Measure execution time

In [18]:
def measure_execution_time(
    driver,
    algorithm,
    graph_name
):

    if algorithm == "louvain":

        query = """
        CALL gds.louvain.stream($graph)
        YIELD nodeId, communityId
        RETURN nodeId, communityId
        """

    elif algorithm == "leiden":

        query = """
        CALL gds.leiden.stream($graph)
        YIELD nodeId, communityId
        RETURN nodeId, communityId
        """

    elif algorithm == "lpa":

        query = """
        CALL gds.labelPropagation.stream($graph)
        YIELD nodeId, communityId
        RETURN nodeId, communityId
        """

    start_time = time.perf_counter()

    with driver.session() as session:
        result = session.run(
            query,
            graph=graph_name
        )

        # Consume the entire result
        result.consume()

    end_time = time.perf_counter()

    execution_time = (
        end_time - start_time
    )

    return execution_time

In [20]:
louvain_time = measure_execution_time(
    driver,
    "louvain",
    "student-friendship-graph"
)

leiden_time = measure_execution_time(
    driver,
    "leiden",
    "student-friendship-graph"
)

lpa_time = measure_execution_time(
    driver,
    "lpa",
    "student-friendship-graph"
)

## the final comparison table

In [21]:
results = pd.DataFrame([
    {
        "Algorithm": "Louvain",
        "Modularity Q": louvain_modularity,
        "Average Conductance": louvain_conductance,
        "Communities": number_of_communities(
            louvain_communities
        ),
        "Average Community Size": avg_louvain,
        "Community Size Std": std_louvain,
        "Execution Time (s)": louvain_time
    },
    {
        "Algorithm": "Leiden",
        "Modularity Q": leiden_modularity,
        "Average Conductance": leiden_conductance,
        "Communities": number_of_communities(
            leiden_communities
        ),
        "Average Community Size": avg_leiden,
        "Community Size Std": std_leiden,
        "Execution Time (s)": leiden_time
    },
    {
        "Algorithm": "Label Propagation",
        "Modularity Q": lpa_modularity,
        "Average Conductance": lpa_conductance,
        "Communities": number_of_communities(
            lpa_communities
        ),
        "Average Community Size": avg_lpa,
        "Community Size Std": std_lpa,
        "Execution Time (s)": lpa_time
    }
])

results

,Algorithm,Modularity Q,Average Conductance,Communities,Average Community Size,Community Size Std,Execution Time (s)
0,Louvain,0.318697,0.623883,18,83.333333,28.146442,3.042581
1,Leiden,0.323979,0.605709,15,100.000000,50.428828,0.233727
2,Label Propagation,0.000000,NaN,1,1500.000000,0.000000,0.112523
